# 02 · XLM-R GPU execution

Run one independently selected English/German document-classification experiment.
The default mode verifies inputs, frozen IDs, tokenizer and GPU without loading
model weights or training. Change `MODE` explicitly to start training.
Detailed upload, persistence and restoration instructions: `experiments/XLMR_README.md`.

This entry point uses a separate output tree and dependency environment from the
classical baseline. Keep both code datasets and notebooks separate on Kaggle.

In [ ]:
import hashlib
import json
import logging
import subprocess
import sys
import tomllib
from pathlib import Path

from IPython.display import display

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
    stream=sys.stdout,
    force=True,
)
logger = logging.getLogger("xlmr_notebook")

MODE = "preflight"
EXPERIMENT = "xlmr_head_only_40k"
MODEL_SCOPE = "sampled"
EVALUATE_HOLDOUT_AFTER_TRAIN = True
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/hm3_xlmr")
CACHE_ROOT = Path("/kaggle/temp/hm3_xlmr_models")
RUNTIME_ROOT = Path("/kaggle/temp/hm3_xlmr_runtime")

if MODE not in {"preflight", "train", "holdout", "refit", "inference"}:
    raise ValueError("Choose one documented explicit mode")
display({"mode": MODE, "experiment": EXPERIMENT, "scope": MODEL_SCOPE})

## Verified inputs and compatible setup

Attach the original competition data and both extracted private XLM-R bundles.
The setup retains the native Torch/CUDA build and installs only pinned auxiliary
packages with `uv`. Exactly one GPU is exposed, including on a dual-T4 runtime.
Python, Torch, CUDA, GPU and auxiliary versions are saved with the run identity.

In [ ]:
markers = [
    path
    for path in INPUT_ROOT.rglob("hm3_bundle.json")
    if json.loads(path.read_text()).get("bundle_kind") == "xlmr_code"
]
if len(markers) != 1:
    raise ValueError("Attach exactly one extracted private XLM-R code bundle")
CODE_ROOT = markers[0].parent
source_record = json.loads(markers[0].read_text())
for name, expected in source_record["files"].items():
    path = (CODE_ROOT / name).resolve()
    if not path.is_relative_to(CODE_ROOT.resolve()):
        raise ValueError("Invalid archive member path")
    with path.open("rb") as stream:
        observed = hashlib.file_digest(stream, "sha256").hexdigest()
    if observed != expected:
        raise ValueError(f"Code checksum mismatch: {name}")
sys.path.insert(0, str(CODE_ROOT / "src"))

from hm3.kaggle_bootstrap import find_bundle, find_competition_inputs
from hm3.xlmr_bootstrap import prepare_xlmr_runtime

SPLIT_ROOT = find_bundle(INPUT_ROOT, "xlmr_split")
configuration = tomllib.loads((CODE_ROOT / "config/xlmr.toml").read_text())
inputs = find_competition_inputs(INPUT_ROOT, configuration["source_sha256"])
PYTHON = prepare_xlmr_runtime(CODE_ROOT, RUNTIME_ROOT)
display({"verified_inputs": inputs, "python": PYTHON, "output": OUTPUT_ROOT})
display(configuration)

## One explicit execution mode

`train` uses the shared 40k development sample and separate 2k development evaluation.
Development macro-F1 selects the checkpoint; the fixed holdout never controls
early stopping. `refit` trains the chosen recipe on all labeled rows in a separate
run. `inference` restores selected weights and exports a CSV without training.
Head-plus-tail is never queued automatically.

In [ ]:
def run_mode(mode: str) -> None:
    command = [
        "env",
        "CUDA_VISIBLE_DEVICES=0",
        str(PYTHON),
        "-I",
        str(CODE_ROOT / "scripts/run_xlmr_kaggle.py"),
        "--code-root",
        str(CODE_ROOT),
        "--split-root",
        str(SPLIT_ROOT),
        "--input-root",
        str(INPUT_ROOT),
        "--output-root",
        str(OUTPUT_ROOT),
        "--cache-root",
        str(CACHE_ROOT),
        "--experiment",
        EXPERIMENT,
        "--scope",
        MODEL_SCOPE,
        "--mode",
        mode,
    ]
    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    ) as process:
        if process.stdout is None:
            raise RuntimeError("Runner output pipe is unavailable")
        for line in process.stdout:
            logger.info("%s", line.rstrip())
        return_code = process.wait()
        if return_code:
            raise subprocess.CalledProcessError(return_code, command)


run_mode(MODE)
if MODE == "train" and EVALUATE_HOLDOUT_AFTER_TRAIN:
    run_mode("holdout")

## Saved evidence

Save the entire run directory using Kaggle output persistence. Temporary runtime
directories and `/kaggle/working` are not durable by themselves. On restoration,
attach one saved run version and retain the same source recipe, runtime and root.
Complete checksummed checkpoints resume optimizer, scheduler, RNG and FP16 state.

In [ ]:
pointer = json.loads((OUTPUT_ROOT / "current_run.json").read_text())
RUN_PATH = OUTPUT_ROOT / pointer["run_directory"]
display(json.loads((RUN_PATH / "run_metadata.json").read_text()))
selection = json.loads((RUN_PATH / "data/metadata/xlmr_development_40k.json").read_text())
display(
    {
        "roles": selection["role_counts"],
        "deviations": selection["distribution_deviations"],
        "missing_evaluation_strata": selection["evaluation_strata_missing"],
        "group_overlap": selection["known_group_overlap"],
        "run_directory": str(RUN_PATH),
    }
)
for name in ["training_complete.json", "holdout_metrics.json"]:
    path = RUN_PATH / name
    if path.exists():
        display(json.loads(path.read_text()))